<a href="https://colab.research.google.com/github/INDHUJA007-HUB/indhuja-day15-workshop/blob/main/Day4/Day4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install pyspark --quiet
print('PySpark installation Successfully!!')

PySpark installation Successfully!!


In [6]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import year,month,to_date,col,round as spark_round
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [7]:
#create spark session
spark=SparkSession.builder\
.appName('Day4_BigData_sales')\
.config('spark.sql.adaptive.enabled','true')\
.getOrCreate()

print(f'Spark Version: {spark.version}')
print(f'Spark Session: ACTIVE')
print(f'Application Name: {spark.sparkContext.appName}')

Spark Version: 4.0.2
Spark Session: ACTIVE
Application Name: Day4_BigData_sales


In [8]:
df_bronze=spark.read\
.option('header','true')\
.option('interSchema','true')\
.csv('/content/drive/MyDrive/Copy of large_sales_data.csv')

print('BRONZE LAYER')
print(f'Rows: {df_bronze.count()}')
print(f'Columns: {len(df_bronze.columns)}')
print(F'Names: {df_bronze.columns}')
print() #just for blank space between lines
df_bronze.printSchema()

BRONZE LAYER
Rows: 5000
Columns: 13
Names: ['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'revenue', 'order_date', 'city', 'region', 'sales_rep', 'payment_method', 'order_status']

root
 |-- order_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- revenue: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)



In [9]:
df_bronze.tail(5)

[Row(order_id='5996', customer_name='Ananya Das', product='Mouse', category='Accessories', quantity='13', unit_price='800', revenue='10400', order_date='2023-11-18', city='Bangalore', region='South', sales_rep='Meera Patel', payment_method='Net Banking', order_status='Cancelled'),
 Row(order_id='5997', customer_name='Suresh Rao', product='Webcam', category='Accessories', quantity='9', unit_price='2500', revenue='22500', order_date='2023-06-07', city='Chennai', region='South', sales_rep='Sunita Rao', payment_method='Credit Card', order_status='Delivered'),
 Row(order_id='5998', customer_name='Arjun Nair', product='Webcam', category='Accessories', quantity='1', unit_price='2500', revenue='2500', order_date='2023-04-07', city='Jaipur', region='North', sales_rep='Kavya Reddy', payment_method='Net Banking', order_status='Cancelled'),
 Row(order_id='5999', customer_name='Arjun Nair', product='Laptop', category='Electronics', quantity='14', unit_price='45000', revenue='630000', order_date='20

In [10]:
df_bronze.write \
.mode('overwrite') \
.parquet('sales_bronze.parquet')
import os

print('Bronze Parquet saved: sales_bronze.parquet')

def get_dir_size(path):
    """Get total size of a file or directory in KB."""
    if os.path.isfile(path):
        return os.path.getsize(path) / 1024
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            total += os.path.getsize(os.path.join(dirpath, f))
    return total / 1024
csv_path = '/content/drive/MyDrive/Copy of large_sales_data.csv'
csv_size = get_dir_size(csv_path)
parquet_size = get_dir_size('sales_bronze.parquet')
reduction = (1 - parquet_size / csv_size) * 100
print(f'\nCSV size : {csv_size:.1f} KB')
print(f'Parquet size : {parquet_size:.1f} KB')
print(f'Reduction : {reduction:.1f}% smaller')
print(f'\nAt 1 TB scale: CSV = 1000 GB --> Parquet = {1000*(1-reduction/100):.0f} GB')

Bronze Parquet saved: sales_bronze.parquet

CSV size : 529.3 KB
Parquet size : 57.4 KB
Reduction : 89.2% smaller

At 1 TB scale: CSV = 1000 GB --> Parquet = 108 GB


In [11]:
print('First 5 rows: ')
df_bronze.show(5,truncate=False)
print('\nBasic statistics for numeric columns:')
df_bronze.select('quantity','unit_price','revenue').describe().show()

First 5 rows: 
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|1001    |Sneha Reddy  |Monitor   |Electronics|12      |22000     |264000 |2023-05-21|Mumbai   |West  |Meera Patel|UPI             |Delivered   |
|1002    |Ramesh Kumar |Printer   |Electronics|10      |12000     |120000 |2023-08-05|Delhi    |North |Anil Sharma|Credit Card     |Shipped     |
|1003    |Rahul Mishra |Mouse     |Accessories|10      |800       |8000   |2023-01-14|Ahmedabad|West  |Meera Patel|Cash on Delivery|Shipped     |
|1004    |Suresh Rao   |Tablet    |Electronics|5       |32000     |160000 |2023-01-04|Surat    |West  |Ravi K

In [12]:
df_silver = df_bronze \
.drop_duplicates() \
.dropna(subset=['order_id','product','revenue'])

df_silver = df_silver \
.withColumn('order_year', year(col('order_date'))) \
.withColumn('order_month', month(col('order_date')))

df_silver = df_silver.withColumn(
    'revenue_category',
    F.when(col('revenue')>40000, 'High')
    .when(col('revenue')>10000, 'Medium')
    .otherwise('Low')
)

print(f"Silver layer rows: {df_silver.count()}")
print(f"New Columns added: order_year, order_month, revenue_category")
df_silver.select('product', 'revenue', 'order_year', 'order_month', 'revenue_category').show(8)

Silver layer rows: 5000
New Columns added: order_year, order_month, revenue_category
+--------+-------+----------+-----------+----------------+
| product|revenue|order_year|order_month|revenue_category|
+--------+-------+----------+-----------+----------------+
| Printer|  12000|      2023|         11|          Medium|
|   Mouse|   8000|      2023|          7|             Low|
|  Webcam|  17500|      2023|         10|          Medium|
|Keyboard|  14400|      2023|          2|          Medium|
| Monitor| 132000|      2023|          8|            High|
|  Laptop| 180000|      2023|          9|            High|
| Speaker|  40500|      2023|          9|            High|
|Keyboard|  18000|      2023|          2|          Medium|
+--------+-------+----------+-----------+----------------+
only showing top 8 rows


In [13]:
df_silver.write \
   .mode('overwrite') \
   .parquet('sales_silver.parquet')

print('Silver Parquet saved: sales_silver.parquet')
print(f'Silver size:{get_dir_size("sales_silver.parquet"):.1f}KB')

#verify by reading it back

df_verify = spark.read.parquet('sales_silver.parquet')
print(f'Rows: {df_verify.count()}(should match Silver count)')
df_verify.printSchema()


Silver Parquet saved: sales_silver.parquet
Silver size:64.1KB
Rows: 5000(should match Silver count)
root
 |-- order_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- revenue: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)
 |-- revenue_category: string (nullable = true)



In [14]:
#silver

top_products=df_silver \
   .groupBy('product') \
   .agg(
       F.sum('revenue').alias('total_revenue'),
       F.count('order_id').alias('num_orders'), \
       F.avg('revenue').alias('avg_order_revenue')
   )\
   .orderBy('total_revenue',ascending=False) \
   .limit(5)

print('=== Top 5 Products by Revenue ===')
top_products.show(truncate=False)

=== Top 5 Products by Revenue ===
+-------+-------------+----------+------------------+
|product|total_revenue|num_orders|avg_order_revenue |
+-------+-------------+----------+------------------+
|Laptop |1.827E8      |502       |363944.22310756973|
|Tablet |1.35104E8    |532       |253954.8872180451 |
|Monitor|8.2126E7     |481       |170740.12474012474|
|Printer|4.4544E7     |488       |91278.68852459016 |
|Speaker|1.6317E7     |470       |34717.02127659575 |
+-------+-------------+----------+------------------+



In [23]:
#silver  made the avg_order_Revenue round as 2 ,and made the total_revenue as ascending
top_products=df_silver \
   .groupBy('product') \
   .agg(
       F.sum('revenue').alias('total_revenue'),
       F.count('order_id').alias('num_orders'), \
       F.round(F.avg('revenue'),2).alias('avg_order_revenue')
   )\
   .orderBy('total_revenue',ascending=True) \
   .limit(5)

print('=== Top 5 Products by Revenue ===')
top_products.show(truncate=False)

=== Top 5 Products by Revenue ===
+----------+-------------+----------+-----------------+
|product   |total_revenue|num_orders|avg_order_revenue|
+----------+-------------+----------+-----------------+
|USB Hub   |2447400.0    |527       |4644.02          |
|Mouse     |3207200.0    |492       |6518.7           |
|Keyboard  |4878000.0    |495       |9854.55          |
|Webcam    |1.09825E7    |532       |20643.8          |
|Headphones|1.35415E7    |481       |28152.81         |
+----------+-------------+----------+-----------------+



In [19]:
#silver region based total_revenue
top_products=df_silver \
   .groupBy('region') \
   .agg(
       F.sum('revenue').alias('total_revenue'),
       F.count('order_id').alias('num_orders'), \
       F.countDistinct('customer_name').alias('unique_customers')
   )\
   .orderBy('total_revenue',ascending=False) \
   .limit(5)

print('=== Top 5 region by Revenue ===')
top_products.show(truncate=False)

=== Top 5 region by Revenue ===
+------+-------------+----------+----------------+
|region|total_revenue|num_orders|unique_customers|
+------+-------------+----------+----------------+
|West  |1.982756E8   |2021      |15              |
|South |1.471459E8   |1483      |15              |
|North |9.98784E7    |995       |15              |
|East  |5.05477E7    |501       |15              |
+------+-------------+----------+----------------+



In [24]:
#silver region based total_revenue
top_products=df_silver \
   .groupBy('order_month') \
   .agg(
       F.round(F.sum('revenue'), 0).cast('long').alias('monthly_revenue'), \
       F.count('order_id').alias('monthly_orders')
   )\
   .orderBy('order_month',ascending=True) \
   .limit(12)

print('=== Monthly revenue report===')
top_products.show(truncate=False)

=== Monthly revenue report===
+-----------+---------------+--------------+
|order_month|monthly_revenue|monthly_orders|
+-----------+---------------+--------------+
|1          |41068200       |423           |
|2          |34485400       |375           |
|3          |40031200       |451           |
|4          |38857100       |390           |
|5          |39984500       |423           |
|6          |40707400       |390           |
|7          |42640700       |405           |
|8          |43718500       |418           |
|9          |37640200       |398           |
|10         |47839000       |479           |
|11         |44577100       |419           |
|12         |44298300       |429           |
+-----------+---------------+--------------+

